# Etapa 4.1 Redis

In [2]:
#!pip install redis pymongo

In [3]:
import redis
# conectamos a la instancia local
r = redis.Redis(host='localhost', port=6379, decode_responses=True)


## 4.1.1 Modelo Clave-Valor y Hashes

### Carrito de compras: 
**Simulación del flujo:** En la siguiente ejecución se demuestra el ciclo de vida del carrito: creación mediante inserción de campos (donde la clave del hash es el ID del plato y el valor es la cantidad), actualización dinámica de cantidades y, finalmente, la destrucción de la clave tras simular la migración exitosa a la base de datos relacional.

In [4]:
id_usuario = 438
clave_carrito = f"carrito:usuario:{id_usuario}"

# limpiamos el carrito por si quedó sucio de una prueba anterior
r.delete(clave_carrito)

print(f"--- Iniciando sesión de compras para el usuario {id_usuario} ---")

# 1. carga de datos: el usuario agrega platos al carrito (hset)
# hset(nombre_hash, campo, valor) -> hset(clave_carrito, id_plato, cantidad)
r.hset(clave_carrito, 105, 2)  # Agrega 2 hamburguesas (plato 105)
r.hset(clave_carrito, 22, 1)   # Agrega 1 porción de papas (plato 22)
r.hset(clave_carrito, 8, 3)    # Agrega 3 gaseosas (plato 8)

print("Se agregaron productos al carrito.")

# 2. consulta general: recuperamos todo el carrito para mostrarlo en la app (hgetall)
carrito_actual = r.hgetall(clave_carrito)
print(f"\nEstado actual del carrito: {carrito_actual}")

# 3. actualización de un campo: el usuario decide que quiere 4 gaseosas en vez de 3
# hincrby es perfecto para carritos porque suma o resta matemáticamente sin tener que leer el valor previo
r.hincrby(clave_carrito, 8, 1) # suma 1 a la cantidad del plato 8

# y el usuario se arrepiente de las papas fritas y las elimina del carrito (hdel)
r.hdel(clave_carrito, 22)

# verificamos los cambios solicitando el valor de un campo específico (hget) y el total
cantidad_gaseosas = r.hget(clave_carrito, 8)
print(f"\nSe actualizó el pedido. Nueva cantidad de gaseosas (plato 8): {cantidad_gaseosas}")

carrito_final = r.hgetall(clave_carrito)
print(f"Carrito final antes de pagar: {carrito_final}")

# 4. simulación de checkout (migración a SQL)
print("\n--- Procesando pago ---")
print("Migrando datos: ejecutando sentencias INSERT INTO pedido y detalle_pedido en PostgreSQL...")

# una vez que el pedido es real en Postgres, el carrito temporal de Redis debe desaparecer
r.delete(clave_carrito)
print("Pago exitoso. Carrito de memoria volátil eliminado.")

--- Iniciando sesión de compras para el usuario 438 ---
Se agregaron productos al carrito.

Estado actual del carrito: {'105': '2', '22': '1', '8': '3'}

Se actualizó el pedido. Nueva cantidad de gaseosas (plato 8): 4
Carrito final antes de pagar: {'105': '2', '8': '4'}

--- Procesando pago ---
Migrando datos: ejecutando sentencias INSERT INTO pedido y detalle_pedido en PostgreSQL...
Pago exitoso. Carrito de memoria volátil eliminado.


### Historial de restaurantes vistos recientemente

**Simulación de flujo:** El siguiente extracto demuestra cómo la lista actúa como una estructura de tamaño fijo (LIFO modificada). A medida que ingresan nuevas interacciones mediante `LPUSH`, los elementos más antiguos son desplazados o eliminados.


In [ ]:

# simulamos la sesión de un usuario
id_usuario = 438
clave_historial = f"historial_vistos:usuario:{id_usuario}"

# limpiamos la clave por si se ejecuta esta celda varias veces seguidas
r.delete(clave_historial)

# 1. el usuario entra a varios restaurantes a lo largo de su sesión
restaurantes_visitados = [15, 3, 27, 8, 15, 42, 9, 33, 11, 5, 18, 22]

print(f"--- Simulando navegación del usuario {id_usuario} ---")
for rest in restaurantes_visitados:
    # lpush inserta el id al principio de la lista (la izquierda)
    r.lpush(clave_historial, rest)
    
    # ltrim corta la lista al instante. le decimos "quédate solo con los índices del 0 al 9"
    # esto garantiza que la lista jamás supere los 10 elementos de longitud
    r.ltrim(clave_historial, 0, 9)
    print(f"El usuario entró al restaurante {rest}")

# 2. consultamos la longitud para verificar que el límite de 10 funcionó perfectamente
longitud = r.llen(clave_historial)
print(f"\nLongitud actual del historial de navegación: {longitud} restaurantes.")

# 3. recuperamos la cinta de restaurantes para mostrarla en la pantalla del celular (lrange)
# el -1 significa "tráeme hasta el final de la lista"
cinta_historial = r.lrange(clave_historial, 0, -1)
print(f"Cinta de 'vistos recientemente' enviada a la interfaz: {cinta_historial}")

# 4. operación de gestión: simulamos que los historiales vencen o el usuario borra el más antiguo
# rpop extrae y elimina el último elemento de la lista (el que está a la derecha de todo)
restaurante_olvidado = r.rpop(clave_historial)
print(f"\nSe eliminó el restaurante más antiguo del historial: {restaurante_olvidado}")

cinta_actualizada = r.lrange(clave_historial, 0, -1)
print(f"Cinta actualizada tras la eliminación: {cinta_actualizada}")

--- Simulando navegación del usuario 438 ---
El usuario entró al restaurante 15
El usuario entró al restaurante 3
El usuario entró al restaurante 27
El usuario entró al restaurante 8
El usuario entró al restaurante 15
El usuario entró al restaurante 42
El usuario entró al restaurante 9
El usuario entró al restaurante 33
El usuario entró al restaurante 11
El usuario entró al restaurante 5
El usuario entró al restaurante 18
El usuario entró al restaurante 22

Longitud actual del historial de navegación: 10 restaurantes.
Cinta de 'vistos recientemente' enviada a la interfaz: ['22', '18', '5', '11', '33', '9', '42', '15', '8', '27']

Se eliminó el restaurante más antiguo del historial: 27
Cinta actualizada tras la eliminación: ['22', '18', '5', '11', '33', '9', '42', '15', '8']
